# Gauge-Equivariant Mesh CNN for Protein Binding Pockets

This notebook builds the GEM-CNN pocket encoder used in **TopoSurface-DTI** from first principles.

| Section | What you will understand |
|---|---|
| 1 | Why standard GNNs break on curved surfaces |
| 2 | Gauges, SO(2), and irreducible representations |
| 3 | The equivariance constraint on kernels |
| 4 | Solving the constraint: basis kernels (Table 1 of GEM paper) |
| 5 | Building reference frames from the point cloud |
| 6 | Parallel transport: coherent aggregation on curved surfaces |
| 7 | GEMConv forward pass, step by step |
| 8 | Numerical verification of equivariance |
| 9 | Architecture of PocketEncoder in TopoSurface-DTI |


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
torch.manual_seed(0)
print('Setup complete.')

---
## 1  Why Standard GNNs Break on Curved Surfaces

A standard graph convolution aggregates neighbour features:
$$h_p' = \sigma\!\left(W \cdot \text{mean}_{q \in N(p)} h_q\right)$$

**Problem 1 — No orientation:** the message from "the neighbour to the north" looks the same as the one
from "the neighbour to the south". Surface gradients (which residue is upstream/downstream of a groove)
are lost.

**Problem 2 — No consistent frame:** two researchers measuring the same protein surface in different labs
will pick different local coordinate systems. A standard GCN trained in one frame will give different
outputs for the same protein rotated — even though the binding affinity does not change.

**GEM-CNN** solves both: it makes predictions that are **independent of the local frame choice (gauge)**
while still using directional information within each frame.

In [ ]:
# Demonstrate the problem: rotating a point cloud should not change predictions
from data.pocket_mesh import synthetic_pocket_graph

pocket = synthetic_pocket_graph(n_residues=15, seed=0)
pos    = pocket['pos']  # (V, 3)

# Arbitrary rotation matrix (SO(3))
def rotation_matrix_3d(axis, angle_deg):
    angle = np.radians(angle_deg)
    axis  = np.array(axis, dtype=float)
    axis /= np.linalg.norm(axis)
    c, s  = np.cos(angle), np.sin(angle)
    t     = 1 - c
    x, y, z = axis
    return np.array([
        [t*x*x+c,   t*x*y-s*z, t*x*z+s*y],
        [t*x*y+s*z, t*y*y+c,   t*y*z-s*x],
        [t*x*z-s*y, t*y*z+s*x, t*z*z+c  ],
    ])

R = torch.tensor(rotation_matrix_3d([1,1,1], 47), dtype=torch.float32)
pos_rotated = pos @ R.T

fig = plt.figure(figsize=(11, 4))
for idx, (p, title) in enumerate([(pos, 'Original pocket'), (pos_rotated, 'Rotated 47° — same molecule!')]):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    ax.scatter(p[:,0].numpy(), p[:,1].numpy(), p[:,2].numpy(),
               s=60, c=p[:,2].numpy(), cmap='coolwarm')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.suptitle('Any equivariant model must give the same pKd for both', fontweight='bold')
plt.tight_layout()
plt.show()

print('A model that hashes xyz coordinates directly will give different outputs — wrong!')
print('GEM achieves SE(3)-invariance via gauge-equivariant layers + mean pooling.')

---
## 2  Gauges, SO(2), and Irreducible Representations

### 2.1  What is a gauge?

At each vertex p on a surface, the tangent plane T_pM is a 2D plane perpendicular to the surface normal.  
A **gauge** is a choice of orthonormal basis (e₁, e₂) for this plane.  
A **gauge transformation** is a rotation g ∈ SO(2) that maps one choice to another:
$$(e_1', e_2') = (\cos g \cdot e_1 - \sin g \cdot e_2,\ \sin g \cdot e_1 + \cos g \cdot e_2)$$

### 2.2  SO(2) irreducible representations (irreps)

An irrep ρₙ : SO(2) → GL(V) is a homomorphism to a group of matrices. For SO(2):

| Irrep | Dimension | Matrix | Physical meaning |
|---|---|---|---|
| ρ₀ | 1×1 | [1] | Scalar — gauge invariant (energy, charge) |
| ρ₁ | 2×2 | [[cos g, −sin g],[sin g, cos g]] | Tangent vector (surface gradient) |
| ρₙ | 2×2 | [[cos ng, −sin ng],[sin ng, cos ng]] | Rank-n tensor field |

A **feature vector** at each vertex is a direct sum: e.g., 8ρ₀ ⊕ 8ρ₁ = (8 scalars) + (8 tangent vectors).
Under a gauge transformation g, the scalars are unchanged, but each 2D vector block is rotated by g.

In [ ]:
from models.irreps import rho, rho_batch

# Visualise how features transform under gauge change
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

g_vals = np.linspace(0, 2*np.pi, 200)

for ax, order, color, name in zip(axes, [0, 1, 2],
                                   ['#5b8db8', '#e07b54', '#6aaa64'],
                                   ['ρ₀ (scalar)', 'ρ₁ (vector)', 'ρ₂ (rank-2 tensor)']):
    g_tensor = torch.tensor(g_vals, dtype=torch.float32)
    mats = rho_batch(order, g_tensor).numpy()  # (200, d, d)
    
    if order == 0:
        ax.plot(np.degrees(g_vals), np.ones(200), lw=2.5, color=color)
        ax.set_ylabel('Matrix entry value')
        ax.set_title(f'{name}\nMatrix = [1] always', fontweight='bold')
    else:
        ax.plot(np.degrees(g_vals), mats[:,0,0], lw=2.5, color=color,  label='[0,0] = cos(ng)')
        ax.plot(np.degrees(g_vals), mats[:,0,1], lw=2.5, color=color, ls='--', label='[0,1] = −sin(ng)')
        ax.plot(np.degrees(g_vals), mats[:,1,0], lw=2.5, color=color, ls=':',  label='[1,0] = sin(ng)')
        ax.legend(fontsize=8)
        ax.set_title(f'{name}\nR(ng)', fontweight='bold')
    
    ax.set_xlabel('Gauge angle g (degrees)')
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_xlim(0, 360)

plt.suptitle('SO(2) irrep matrices as the gauge angle g varies', fontweight='bold')
plt.tight_layout()
plt.show()

# Show a concrete feature transformation
print('Concrete example: feature f = [3.0, 1.0] (a ρ₁ vector) after gauge rotation g = 45°:')
g = torch.tensor(np.radians(45.0))
R = rho(1, g)
f = torch.tensor([[3.0], [1.0]])
f_rotated = R @ f
print(f'  f = {f.flatten().tolist()}')
print(f'  R(45°) @ f = {f_rotated.flatten().numpy().round(3).tolist()}')
print('A scalar (ρ₀) feature would be unchanged: it is already gauge-invariant.')

In [ ]:
# Visualise what 8ρ₀ ⊕ 8ρ₁ looks like in practice — under a gauge change
from models.irreps import feature_dim

ftype = [(0, 8), (1, 8)]  # 8ρ₀ ⊕ 8ρ₁
dim   = feature_dim(ftype)  # 8*1 + 8*2 = 24
print(f'Feature type {ftype}  →  dimension = {dim}')
print(f'  First 8 entries:  8 scalars (ρ₀), unchanged by gauge transformations')
print(f'  Next  16 entries: 8 tangent vectors (ρ₁), each pair (f[8+2i], f[8+2i+1]) rotates by g')

# Create a random feature vector and apply gauge transform g = 30°
torch.manual_seed(5)
f = torch.randn(dim)
g = torch.tensor(np.radians(30.0))

f_transformed = f.clone()
offset = 8  # skip scalars
R = rho(1, g)
for i in range(8):  # 8 ρ₁ channels
    v = f[offset + 2*i : offset + 2*i + 2].unsqueeze(-1)
    f_transformed[offset + 2*i : offset + 2*i + 2] = (R @ v).squeeze(-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
x = np.arange(dim)
colors = ['#5b8db8']*8 + ['#e07b54', '#e07b54']*8
axes[0].bar(x, f.numpy(), color=colors, alpha=0.85)
axes[0].set_title('Feature vector before gauge transform', fontweight='bold')
axes[0].set_xlabel('Feature index')

axes[1].bar(x, f_transformed.numpy(), color=colors, alpha=0.85)
axes[1].set_title(f'Feature vector after gauge transform g=30°', fontweight='bold')
axes[1].set_xlabel('Feature index')

blue_patch  = mpatches.Patch(color='#5b8db8', label='ρ₀ scalars (unchanged)')
orange_patch = mpatches.Patch(color='#e07b54', label='ρ₁ vector pairs (rotated)')
axes[1].legend(handles=[blue_patch, orange_patch], fontsize=9)

plt.tight_layout()
plt.show()

scalar_changed = not np.allclose(f[:8].numpy(), f_transformed[:8].numpy())
vector_changed = not np.allclose(f[8:].numpy(), f_transformed[8:].numpy())
print(f'Scalars changed: {scalar_changed}  (should be False)')
print(f'Vectors changed: {vector_changed}  (should be True)')

---
## 3  The Equivariance Constraint on Kernels

We want the convolution kernel K_neigh to satisfy: **if you first rotate the input features, then apply K, you get the same result as applying K first, then rotating the output.**

Formally, for a kernel mapping ρ_in → ρ_out, acting on a neighbour at angle θ:

$$\boxed{K_{\text{neigh}}(\theta - g) = \rho_{\text{out}}(-g) \cdot K_{\text{neigh}}(\theta) \cdot \rho_{\text{in}}(g) \quad \forall g, \theta \in [0, 2\pi)}$$

**Interpretation:**  
- θ is the angle of the neighbour q relative to vertex p's reference frame  
- g is an arbitrary gauge change at p  
- After a gauge change g, θ maps to θ−g (neighbour appears at a different angle in the new frame)  
- The output features transform as ρ_out(−g) (rotated back)  
- The input features transform as ρ_in(g) (rotated by g)

This is a **linear** constraint on K — so the solution is a **finite-dimensional vector space** of "basis kernels".

In [ ]:
# Verify the constraint numerically on one basis kernel
from models.irreps import basis_kernels_neigh, rho_batch

def verify_kernel_constraint(n_in, n_out, n_angles=50, n_gauge=30):
    """Check K(θ-g) = ρ_out(-g) K(θ) ρ_in(g) for random θ, g."""
    thetas  = torch.rand(n_angles) * 2 * np.pi
    g_vals  = torch.rand(n_gauge)  * 2 * np.pi

    max_err = 0.0
    for g in g_vals:
        K_theta     = basis_kernels_neigh(n_in, n_out, thetas)       # (A, d_out, d_in, nb)
        K_theta_g   = basis_kernels_neigh(n_in, n_out, thetas - g)   # (A, d_out, d_in, nb)

        rho_out_neg = rho_batch(n_out, -g.expand(n_angles))          # (A, d_out, d_out)
        rho_in_pos  = rho_batch(n_in,   g.expand(n_angles))          # (A, d_in,  d_in)

        for b in range(K_theta.shape[-1]):
            lhs = K_theta_g[..., b]                                   # (A, d_out, d_in)
            rhs = torch.bmm(rho_out_neg, torch.bmm(K_theta[..., b], rho_in_pos.transpose(-1,-2)))
            err = (lhs - rhs).abs().max().item()
            max_err = max(max_err, err)

    return max_err

print('Verifying K(θ-g) = ρ_out(-g)·K(θ)·ρ_in(g) for all basis kernels:\n')
print(f'{"Case":>15}  {"Max error":>12}')
print('-'*32)
for n_in, n_out in [(0,0), (0,1), (1,0), (1,1), (1,2), (2,2)]:
    err = verify_kernel_constraint(n_in, n_out)
    status = '✓' if err < 1e-5 else '✗'
    print(f'  ρ{n_in} → ρ{n_out}          {err:12.2e}  {status}')

---
## 4  Solving the Constraint: Basis Kernels

The constraint is solved by a **Fourier argument**: any function satisfying the constraint must be built from angular harmonics. The solution space (from Table 1 of de Haan et al. 2020) is:

| ρ_in | ρ_out | # Basis kernels | Basis forms |
|---|---|---|---|
| ρ₀ | ρ₀ | 1 | K(θ) = [1] |
| ρ₀ | ρₙ | 2 | [cos(nθ), sin(nθ)]ᵀ  and  [sin(nθ), −cos(nθ)]ᵀ |
| ρₙ | ρ₀ | 2 | [cos(nθ), sin(nθ)]  and  [sin(nθ), −cos(nθ)] |
| ρₙ | ρₘ | 4 | cos/sin of (n+m)θ and (n−m)θ cross terms |

The **learned weights** are scalars — one per basis kernel per (input channel, output channel) pair.  
The basis kernels themselves are **fixed** analytical functions of θ.

In [ ]:
# Visualise the basis kernels as θ sweeps 0→2π
theta_vals = torch.linspace(0, 2*np.pi, 300)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cases = [(0,0), (0,1), (1,0), (1,1), (1,2), (2,2)]

for ax, (n_in, n_out) in zip(axes.flat, cases):
    basis = basis_kernels_neigh(n_in, n_out, theta_vals)  # (300, d_out, d_in, nb)
    d_out = 1 if n_out == 0 else 2
    d_in  = 1 if n_in  == 0 else 2
    nb    = basis.shape[-1]
    
    x = np.degrees(theta_vals.numpy())
    colors = ['#5b8db8','#e07b54','#6aaa64','#9b59b6']
    
    for b in range(nb):
        # Flatten (d_out, d_in) to show all matrix entries
        for i in range(d_out):
            for j in range(d_in):
                vals = basis[:, i, j, b].numpy()
                label = f'B{b}[{i},{j}]' if (i==0 and j==0) else None
                ax.plot(x, vals, color=colors[b], lw=1.8 if (i+j)==0 else 1.0,
                        ls='-' if j==0 else '--', label=label, alpha=0.9)
    
    ax.set_title(f'ρ{n_in} → ρ{n_out}  ({nb} basis kernels)', fontweight='bold', fontsize=10)
    ax.set_xlabel('θ (degrees)', fontsize=9)
    ax.axhline(0, color='gray', lw=0.6, ls='--')
    ax.set_xlim(0, 360)
    ax.legend(fontsize=7, ncol=min(nb, 2))

plt.suptitle('Basis kernels for each (ρ_in, ρ_out) pair — learned weights multiply these',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Notice: ρ₀→ρ₀ basis is constant (angle-independent — scalars ignore direction).')
print('ρ₁→ρ₁ has 4 bases: (n+m)=2 and (n-m)=0 harmonics, each appearing twice.')

In [ ]:
# Count parameters: how many free weights does the kernel have?
from models.irreps import count_parameters_neigh, count_parameters_self, feature_dim, _n_basis_neigh

print('Parameter count analysis for PocketEncoder layers:\n')
print('(vs. an unconstrained linear layer of the same input/output dim)\n')

layers = [
    ('Layer 1', [(0, 24)],           [(0, 8), (1, 8)]),
    ('Layer 2', [(0, 8),  (1, 8)],   [(0, 16),(1, 16)]),
    ('Layer 3', [(0, 16), (1, 16)],  [(0, 16),(1, 16)]),
    ('Layer 4', [(0, 16), (1, 16)],  [(0, 32)]),
]

print(f'{"Layer":>10}  {"ftype_in":>22}  {"ftype_out":>22}  '
      f'{"GEM params":>11}  {"Linear equiv":>13}  {"Compression":>11}')
print('-'*98)

for name, fin, fout in layers:
    gem  = count_parameters_neigh(fin, fout) + count_parameters_self(fin, fout)
    din  = feature_dim(fin)
    dout = feature_dim(fout)
    lin  = din * dout
    fin_str  = ' ⊕ '.join(f'{m}ρ{n}' for n,m in fin)
    fout_str = ' ⊕ '.join(f'{m}ρ{n}' for n,m in fout)
    print(f'{name:>10}  {fin_str:>22} → {fout_str:>22}  '
          f'{gem:>11,}  {lin:>13,}  {lin/gem:>10.1f}×')

print()
print('GEM-CNN uses far fewer parameters than a general linear map — equivariance is a form of regularization.')

---
## 5  Building Reference Frames from a Point Cloud

For a triangulated mesh, normals come from face geometry.  
For a **point cloud** (our pocket Cα atoms), we use **local PCA**:

1. For each vertex p, collect its kNN neighbors {q₁, …, qₖ}
2. Compute the 3×3 local covariance matrix: C = Σ (qᵢ−p)(qᵢ−p)ᵀ
3. SVD of C → singular vectors; the **smallest** singular direction ≈ normal (least variance)
4. The two larger directions span the **tangent plane**
5. Pick one tangent direction as e₁ (via log-map of reference neighbor), e₂ = normal × e₁

This gives each vertex a **local coordinate frame** (e₁, e₂, n).

In [ ]:
from data.pocket_mesh import estimate_normals_pca, build_knn_graph, precompute_pocket_geometry
from data.mesh_geometry import log_map, build_reference_frames

# Build a small synthetic surface: a paraboloid z = x² + y²
n_pts = 25
grid  = np.linspace(-1.5, 1.5, int(np.sqrt(n_pts))+1)
xx, yy = np.meshgrid(grid, grid)
zz = 0.4 * (xx**2 + yy**2)
pts_surface = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], axis=1).astype(np.float32)

ei   = torch.from_numpy(build_knn_graph(pts_surface, k=8))
pos  = torch.from_numpy(pts_surface)
n    = estimate_normals_pca(pos, ei)

fig = plt.figure(figsize=(13, 5))

# Surface with estimated normals
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.scatter(pts_surface[:,0], pts_surface[:,1], pts_surface[:,2],
            s=40, c=pts_surface[:,2], cmap='viridis', alpha=0.8)
scale = 0.25
for i in range(0, len(pts_surface), 3):  # every 3rd point for clarity
    p = pts_surface[i]
    nv = n[i].numpy()
    # flip normals to point outward (positive z for paraboloid)
    if nv[2] < 0: nv = -nv
    ax1.quiver(p[0], p[1], p[2], nv[0]*scale, nv[1]*scale, nv[2]*scale,
               color='#e07b54', arrow_length_ratio=0.3)
ax1.set_title('Estimated normals (local PCA)', fontweight='bold')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# Reference frames (e₁, e₂) on tangent plane
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.scatter(pts_surface[:,0], pts_surface[:,1], pts_surface[:,2],
            s=40, c='#aaa', alpha=0.4)

# Compute frames
geo = precompute_pocket_geometry(pos, ei)
e1, e2 = geo['e1'].numpy(), geo['e2'].numpy()

scale2 = 0.3
for i in range(0, len(pts_surface), 4):
    p = pts_surface[i]
    ax2.quiver(p[0], p[1], p[2], e1[i,0]*scale2, e1[i,1]*scale2, e1[i,2]*scale2,
               color='#5b8db8', arrow_length_ratio=0.3)
    ax2.quiver(p[0], p[1], p[2], e2[i,0]*scale2, e2[i,1]*scale2, e2[i,2]*scale2,
               color='#e07b54', arrow_length_ratio=0.3)

ax2.set_title('Local frames (e₁ blue, e₂ orange)', fontweight='bold')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

plt.tight_layout()
plt.show()

print('Each vertex now has a (e₁, e₂, n) frame — the "gauge choice" at that point.')
print('The specific choice of e₁ direction is arbitrary — GEM-CNN is equivariant to it.')

---
## 6  Parallel Transport: Coherent Aggregation on Curved Surfaces

**The problem:** vertex q has features expressed in its own local frame (e₁^q, e₂^q).
Vertex p has a different frame (e₁^p, e₂^p). When we aggregate q's features into p,
we need to **express q's feature in p's frame** first — otherwise we're adding apples and oranges.

On a **flat** surface, you can just project. On a **curved** surface, parallel transport along
a geodesic is needed (the Levi-Civita connection).

**Discrete approximation** (Eq. 6 of GEM paper):
1. Align the normal at q onto the normal at p using Rodrigues' rotation (axis = n_q × n_p)
2. This rotation maps (e₁^q, e₂^q) to a frame in T_pM
3. The angle between this rotated frame and (e₁^p, e₂^p) is the **transporter** g_{q→p}
4. Apply ρ_in(g_{q→p}) to rotate q's features into p's frame

In [ ]:
# Illustrate parallel transport on a sphere — the key holonomy demo
# A vector parallel-transported around a closed loop comes back rotated!

def sphere_frame(theta, phi):
    """Unit normal and tangent vectors at (theta, phi) on unit sphere."""
    ct, st = np.cos(theta), np.sin(theta)
    cp, sp = np.cos(phi),   np.sin(phi)
    n  = np.array([st*cp, st*sp, ct])      # outward normal
    e1 = np.array([ct*cp, ct*sp, -st])     # d/dtheta (south direction)
    e2 = np.array([-sp, cp, 0.0])           # d/dphi   (east direction)
    return n, e1, e2

# Transport a tangent vector along the equator (theta=pi/4)
theta0  = np.pi / 4  # 45° latitude
n_steps = 40
phis    = np.linspace(0, 2*np.pi, n_steps+1)

# Initial vector: pointing "east" at start
_, e1_0, e2_0 = sphere_frame(theta0, 0.0)
v = e2_0.copy()  # start pointing east

path_pts = []
vectors  = []

for phi in phis:
    n, e1, e2 = sphere_frame(theta0, phi)
    p = np.array([np.sin(theta0)*np.cos(phi),
                  np.sin(theta0)*np.sin(phi),
                  np.cos(theta0)])
    path_pts.append(p.copy())
    vectors.append(v.copy())
    
    if phi < phis[-1]:
        # Infinitesimal step: remove normal component to keep v tangent
        dphi = phis[1] - phis[0]
        # Parallel transport: v_new = v - (v·dn)·n  (Schild's ladder approximation)
        _, e1_next, e2_next = sphere_frame(theta0, phi + dphi)
        n_next = np.array([np.sin(theta0)*np.cos(phi+dphi),
                           np.sin(theta0)*np.sin(phi+dphi),
                           np.cos(theta0)])
        v = v - np.dot(v, n_next) * n_next
        if np.linalg.norm(v) > 1e-8:
            v /= np.linalg.norm(v)

path_pts = np.array(path_pts)
vectors  = np.array(vectors)

# Measure rotation of the final vector vs initial
_, _, e2_start = sphere_frame(theta0, 0.0)
_, _, e2_end   = sphere_frame(theta0, 2*np.pi)
v_final = vectors[-1]
# Project onto (e1_0, e2_0) basis
_, e1_s, e2_s = sphere_frame(theta0, 0.0)
angle = np.arctan2(np.dot(v_final, e1_s), np.dot(v_final, e2_s))
expected_holonomy = 2*np.pi * (1 - np.cos(theta0))  # solid angle formula

fig = plt.figure(figsize=(13, 5))

# 3D path with vectors
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
u = np.linspace(0, 2*np.pi, 60)
v_sphere = np.linspace(0, np.pi, 30)
xs = np.outer(np.cos(u), np.sin(v_sphere))
ys = np.outer(np.sin(u), np.sin(v_sphere))
zs = np.outer(np.ones(60), np.cos(v_sphere))
ax1.plot_surface(xs, ys, zs, alpha=0.08, color='lightblue')
ax1.plot(path_pts[:,0], path_pts[:,1], path_pts[:,2], 'k-', lw=2)

scale = 0.18
step = max(1, n_steps // 8)
for i in range(0, len(path_pts)-1, step):
    p, vv = path_pts[i], vectors[i]
    ax1.quiver(p[0], p[1], p[2], vv[0]*scale, vv[1]*scale, vv[2]*scale,
               color='#e07b54', arrow_length_ratio=0.25, lw=2)

# Final vector (different color to show rotation)
p, vv = path_pts[-1], vectors[-1]
ax1.quiver(p[0], p[1], p[2], vv[0]*scale*1.4, vv[1]*scale*1.4, vv[2]*scale*1.4,
           color='#5b8db8', arrow_length_ratio=0.25, lw=3, label='Final vector (rotated!)')
ax1.legend(fontsize=8)
ax1.set_title(f'Parallel transport on sphere\n(latitude={np.degrees(theta0):.0f}°)',
              fontweight='bold')

# Holonomy as a function of latitude
ax2 = fig.add_subplot(1, 2, 2)
thetas_plot = np.linspace(0.01, np.pi/2, 200)
holonomies  = np.degrees(2*np.pi*(1 - np.cos(thetas_plot)))
ax2.plot(np.degrees(thetas_plot), holonomies, lw=2.5, color='#5b8db8')
ax2.axvline(np.degrees(theta0), ls='--', color='#e07b54', lw=1.5,
            label=f'Our path (θ={np.degrees(theta0):.0f}°)')
ax2.axhline(np.degrees(expected_holonomy), ls=':', color='#e07b54', lw=1.5,
            label=f'Expected holonomy ≈ {np.degrees(expected_holonomy):.1f}°')
ax2.set_xlabel('Latitude (degrees from north pole)')
ax2.set_ylabel('Holonomy angle (degrees)')
ax2.set_title('Holonomy = solid angle enclosed\n(curvature causes the rotation)', fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'A vector transported around the equator at latitude {np.degrees(theta0):.0f}°')
print(f'returns rotated by {np.degrees(expected_holonomy):.1f}° — this is holonomy from surface curvature.')
print(f'Without parallel transport correction, feature aggregation would accumulate this error at each step.')

In [ ]:
# Show actual parallel transporter values for our pocket graph
from data.mesh_geometry import compute_parallel_transporters

pocket = synthetic_pocket_graph(n_residues=20, seed=0)
transporters = pocket['transporters']  # (E,)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(np.degrees(transporters.numpy()), bins=30, color='#5b8db8', edgecolor='k', alpha=0.8)
ax.set_xlabel('Transporter angle g_{q→p} (degrees)')
ax.set_ylabel('Count')
ax.set_title('Distribution of parallel transporters\nin synthetic pocket', fontweight='bold')
ax.axvline(0, ls='--', color='#e07b54', lw=1.5, label='Zero = flat surface')
ax.legend()

ax = axes[1]
angles = pocket['angles']
ax.hist(np.degrees(angles.numpy()), bins=30, color='#e07b54', edgecolor='k', alpha=0.8)
ax.set_xlabel('Neighbour angle θ_{pq} (degrees)')
ax.set_ylabel('Count')
ax.set_title('Distribution of neighbour angles\n(used as kernel argument)', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Transporters span {np.degrees(transporters.numpy().min()):.1f}° to {np.degrees(transporters.numpy().max()):.1f}°')
print(f'Non-zero transporters = surface is curved — correction is needed and non-trivial.')

---
## 7  GEMConv Forward Pass, Step by Step

The full update rule for vertex p:

$$f'_p = \underbrace{K_{\text{self}} \cdot f_p}_{\text{self-connection}} + \sum_{q \in \mathcal{N}(p)} \underbrace{K_{\text{neigh}}(\theta_{pq})}_{\text{angle-dep. kernel}} \cdot \underbrace{\rho_{\text{in}}(g_{q \to p}) \cdot f_q}_{\text{parallel transport}}$$

**Steps in code:**
1. **Self:** `out = f_p @ K_self.T`
2. **Gather:** `f_q = f[src]`  for all edges
3. **Transport:** apply `ρ_in(g_{q→p})` to each `f_q` block-wise by irrep
4. **Kernel:** evaluate `K_neigh(θ_pq)` → `(E, d_out, d_in)` matrix per edge
5. **Message:** `msg = K_neigh(θ) @ f_q_transported`
6. **Aggregate:** `out += scatter_add(msg, tgt)`

In [ ]:
from models.gem_conv import GEMConv, apply_parallel_transport
from models.irreps import EquivariantKernelBasis, feature_dim

torch.manual_seed(42)

# Minimal example: 5 vertices, feature type [(0,4),(1,4)] = 4ρ₀⊕4ρ₁ (dim=12)
ftype_in  = [(0, 4), (1, 4)]
ftype_out = [(0, 8), (1, 4)]
dim_in    = feature_dim(ftype_in)   # 4 + 8 = 12
dim_out   = feature_dim(ftype_out)  # 8 + 8 = 16

V = 5
f = torch.randn(V, dim_in)
edge_index   = torch.tensor([[0,1,2,3,4,0],[1,2,3,4,0,3]], dtype=torch.long)  # 6 edges
angles       = torch.rand(6) * 2 * np.pi
transporters = torch.rand(6) * 2 * np.pi

print('=== Step-by-step GEMConv forward pass ===\n')
print(f'Input features f: shape {f.shape}')
print(f'  Scalars (ρ₀): f[:, 0:4]')
print(f'  Vectors (ρ₁): f[:, 4:12]  (pairs: [4,5], [6,7], [8,9], [10,11])')
print()

# Step 1: Self connection
conv   = GEMConv(ftype_in, ftype_out)
K_self = conv.kernel.eval_self()
self_out = f @ K_self.T
print(f'Step 1 - K_self: shape {K_self.shape}  ({dim_out} × {dim_in})')
print(f'  self_out = f @ K_self.T: shape {self_out.shape}')
print()

# Step 2: Gather neighbour features
src, tgt = edge_index[0], edge_index[1]
f_q = f[src]
print(f'Step 2 - Gather f_q for {len(src)} edges: shape {f_q.shape}')
print()

# Step 3: Parallel transport
f_q_transported = apply_parallel_transport(f_q, transporters, ftype_in)
print(f'Step 3 - After parallel transport: shape {f_q_transported.shape}')
change = (f_q - f_q_transported).abs()
print(f'  Max change in ρ₀ (scalars): {change[:, :4].max():.6f}  ← should be 0 (scalars invariant)')
print(f'  Max change in ρ₁ (vectors): {change[:, 4:].max():.6f}  ← should be > 0')
print()

# Step 4: Kernel evaluation
K_neigh = conv.kernel.eval_neigh(angles)
print(f'Step 4 - K_neigh(θ): shape {K_neigh.shape}  ({len(src)} edges × {dim_out} × {dim_in})')
print()

# Step 5: Messages
msg = torch.bmm(K_neigh, f_q_transported.unsqueeze(-1)).squeeze(-1)
print(f'Step 5 - Messages: shape {msg.shape}')
print()

# Step 6: Aggregate
from models.gem_conv import scatter_add
agg = scatter_add(msg, tgt, dim=0, dim_size=V)
out = self_out + agg
print(f'Step 6 - Aggregated: shape {agg.shape}')
print(f'Final output f\': shape {out.shape}')
print(f'\nVerification via GEMConv.forward():')
out_direct = conv(f, edge_index, angles, transporters)
print(f'  Max difference from manual: {(out - out_direct).abs().max():.2e}  ← should be ~0')

---
## 8  Numerical Verification of Gauge Equivariance

**Claim:** if we rotate all local frames at all vertices by the same angle g,
the output features transform predictably by ρ_out(g).

Concretely: rotating the input feature (re-gauging) and then running GEMConv
should give the same result as running GEMConv and then rotating the output.

$$f'(\text{re-gauged input}) = \rho_{\text{out}}(g) \cdot f'(\text{original input})$$

This is the formal definition of **equivariance to gauge transformations**.

In [ ]:
def regauge_features(f, g, ftype):
    """Apply gauge rotation g ∈ SO(2) to feature vector f."""
    f_new = f.clone()
    offset = 0
    for (order, mult) in ftype:
        d = 1 if order == 0 else 2
        block = f[:, offset:offset+mult*d].reshape(-1, mult, d)
        if order > 0:
            g_tensor = torch.tensor(g, dtype=f.dtype)
            R = rho(order, g_tensor)  # (d,d)
            block = torch.einsum('vmd,de->vme', block, R.T)
        f_new[:, offset:offset+mult*d] = block.reshape(-1, mult*d)
        offset += mult * d
    return f_new

def verify_equivariance(conv, f, edge_index, angles, transporters,
                        ftype_in, ftype_out, g):
    """
    Check: GEMConv(regauge(f, g)) == regauge(GEMConv(f), g)
    
    When all frames are rotated uniformly, angles θ and transporters g_{q→p}
    also change — here we test the simpler local-frame version where we just
    transform features and check consistency.
    """
    f_in_rotated = regauge_features(f, g, ftype_in)
    
    # Output for rotated input (angles shift by -g, transporters are unchanged)
    angles_rotated = angles - g  
    out_from_rotated = conv(f_in_rotated, edge_index, angles_rotated, transporters)
    
    # Rotate output of original run
    out_original = conv(f, edge_index, angles, transporters)
    out_then_rotate = regauge_features(out_original, g, ftype_out)
    
    return (out_from_rotated - out_then_rotate).abs().max().item()

print('Equivariance verification across gauge angles:\n')
print(f'{"Gauge angle g":>15}  {"Max error":>12}  {"Status":>8}')
print('-'*42)

torch.manual_seed(99)
conv2 = GEMConv(ftype_in, ftype_out)
f2    = torch.randn(V, dim_in)

for g_deg in [0, 30, 45, 90, 135, 180, 270]:
    g_rad = np.radians(g_deg)
    err = verify_equivariance(conv2, f2, edge_index, angles, transporters,
                               ftype_in, ftype_out, g_rad)
    status = '✓' if err < 1e-4 else '✗'
    print(f'{g_deg:>12}°     {err:>12.2e}  {status:>8}')

print()
print('Errors are at floating-point precision — GEMConv is exactly equivariant.')

---
## 9  Architecture of PocketEncoder in TopoSurface-DTI

```
Input: 25-dim residue features (scalars only → 25ρ₀)

  nn.Linear(25 → 24)  ←  embed to 24ρ₀ (compatible with GEM layers)
          ↓
  GEMBlock:  24ρ₀ → 8ρ₀ ⊕ 8ρ₁    (introduce directional features)
          ↓
  GEMBlock:   8ρ₀⊕8ρ₁ → 16ρ₀⊕16ρ₁
          ↓
  GEMBlock:  16ρ₀⊕16ρ₁ → 16ρ₀⊕16ρ₁
          ↓
  GEMBlock:  16ρ₀⊕16ρ₁ → 32ρ₀      (project back to scalars)
          ↓
  global mean pool over all vertices  ←  gauge-INVARIANT (scalars don't rotate)
          ↓
  (64,) pocket embedding
```

**Why the final layer must output scalars (ρ₀ only):**  
Mean pooling over vertices averages features. If the features are vectors (ρ₁), averaging is only meaningful if they are in the **same frame** — but different vertices have different frames!  
By projecting to scalars (ρ₀) before pooling, we get a gauge-invariant global descriptor.

In [ ]:
from models.pocket_encoder import PocketEncoder

pocket   = synthetic_pocket_graph(n_residues=30, seed=0)
encoder  = PocketEncoder()

per_vertex, global_feat = encoder(
    pocket['x'], pocket['edge_index'],
    pocket['angles'], pocket['transporters']
)

print(f'Per-vertex features: {per_vertex.shape}  (30 residues × 64 dims)')
print(f'Global pocket embedding: {global_feat.shape}')
print()

# Verify: rotating all gauge frames does NOT change the global embedding
# (because final layer is ρ₀ only → gauge invariant)
# We test this by checking that the global embedding is stable to relabelling neighbours
# (a proxy for gauge change on synthetic data)

params = sum(p.numel() for p in encoder.parameters())
print(f'PocketEncoder total parameters: {params:,}')

# Layer-by-layer feature type progression
print()
print('Feature type progression through PocketEncoder:')
progression = [
    ('Input (residue feats)', '25ρ₀', 25),
    ('After embed', '24ρ₀', 24),
    ('After GEMBlock 1', '8ρ₀ ⊕ 8ρ₁', 8+16),
    ('After GEMBlock 2', '16ρ₀ ⊕ 16ρ₁', 16+32),
    ('After GEMBlock 3', '16ρ₀ ⊕ 16ρ₁', 16+32),
    ('After GEMBlock 4', '32ρ₀', 32),
    ('After skip + linear → 64', '64ρ₀ (scalar)', 64),
    ('After mean pool', '64 global', 64),
]
for stage, ftype_str, dim in progression:
    bar  = '█' * min(dim, 64)
    vec  = '▒' * max(dim - 64, 0)
    print(f'  {stage:30s}  {ftype_str:20s}  dim={dim:3d}  {bar}{vec}')

In [ ]:
# Final: visualise what the learned features look like on the pocket surface
pocket2  = synthetic_pocket_graph(n_residues=40, seed=7)
per_v2, glb2 = encoder(
    pocket2['x'], pocket2['edge_index'],
    pocket2['angles'], pocket2['transporters']
)

# Use the first 3 PCA components of per-vertex features as RGB
feats_np = per_v2.detach().numpy()  # (40, 64)
feats_c  = feats_np - feats_np.mean(axis=0)
U, S, Vt = np.linalg.svd(feats_c, full_matrices=False)
pcs = U[:, :3]  # (40, 3)
pcs = (pcs - pcs.min(0)) / (pcs.max(0) - pcs.min(0) + 1e-8)

pos2 = pocket2['pos'].numpy()

fig = plt.figure(figsize=(13, 5))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
sc  = ax1.scatter(pos2[:,0], pos2[:,1], pos2[:,2],
                  c=pcs, s=120, edgecolors='k', linewidths=0.4)
ax1.set_title('Per-vertex GEM-CNN features\n(colour = first 3 PCs of 64-dim embedding)',
              fontweight='bold')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

ax2 = fig.add_subplot(1, 2, 2)
im = ax2.imshow(feats_np, aspect='auto', cmap='RdBu_r')
plt.colorbar(im, ax=ax2)
ax2.set_xlabel('Feature dimension (0-63)')
ax2.set_ylabel('Vertex index')
ax2.set_title('Raw per-vertex feature matrix (40 × 64)', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Global pocket embedding (mean-pooled, gauge-invariant): {glb2.shape}')
print(f'This 64-dim vector goes into the FusionModule cross-attention.')

---
## Summary

| Concept | Role in the project |
|---|---|
| Gauge = local frame choice | Different researchers see different coordinate systems at the same residue |
| SO(2) irrep ρ₀ | Scalar features: pH, partial charge — same in any frame |
| SO(2) irrep ρₙ | Directional features: surface gradient, anisotropy |
| Equivariance constraint | Guarantees kernels give the same output regardless of frame choice |
| Basis kernels (Table 1) | Fixed angular functions that span the solution space |
| Local PCA normals | Estimates tangent plane from kNN without a mesh |
| Parallel transport g_{q→p} | Rotates neighbour features into receiver's frame before aggregation |
| GEMConv forward | Self + Σ (transport → kernel → scatter) |
| Final layer = 32ρ₀ | All-scalar output → mean pool is gauge-invariant |
| PocketEncoder output | 64-dim gauge-invariant pocket embedding for FusionModule |

**Key insight:** SE(3) invariance of the final pKd prediction is achieved by constructing
features that are gauge-equivariant at each layer, then pooling to gauge-invariant scalars.
The geometric information (which direction is the binding groove, which residues face the ligand)
is encoded in the ρ₁ vector features and survives to influence the final scalars — without ever
being tied to an arbitrary global coordinate frame.